# 01 - Create Pseudobulks

This notebook loads the single-cell/nucleus reference data, removes unwanted genes, creates the cell-level train/test split, and simulates train/test pseudobulks.

Important: `train_adata` and `test_adata` are created here by `split_cells_by_type()`. They are then used directly by `make_pseudobulks()` so the model is evaluated on pseudobulks built from held-out cells.


In [36]:
from __future__ import annotations

import importlib.util
from pathlib import Path
import json
from typing import Iterable, Sequence

REQUIRED_PACKAGES = {
    "anndata": "anndata",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
missing_packages = [name for name, module in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    raise ImportError(
        "Missing required packages: "
        + ", ".join(missing_packages)
        + ". Install the project environment first, for example with `uv sync`."
    )

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root, where pyproject.toml is located.")

DATA_DIR = PROJECT_ROOT / "data" / "train_datasets" / "rat_ref"
COUNTS_PATH = DATA_DIR / "counts.h5ad"
METADATA_PATH = DATA_DIR / "metadata_clean.csv"
SC_DE_GENE_PATH = PROJECT_ROOT / "data" / "misc" / "empty.json"
SN_DE_GENE_PATH = PROJECT_ROOT / "data" / "misc" / "empty.json"
COMMON_BULK_GENES_PATH = PROJECT_ROOT / "data" / "misc" / "common_bulk_genes.json"

REFERENCE_LABEL = "rat_sub_std"

REFERENCE_DIR = PROJECT_ROOT / "data" / "references" / REFERENCE_LABEL
PSEUDOBULK_DIR = REFERENCE_DIR / "pseudobulk"
PLOTS_DIR = PSEUDOBULK_DIR / "plots"

CELL_ID_COLUMN = "cellID"
CELLTYPE_COLUMN = "celltype" if "main" in REFERENCE_LABEL else "sub_celltype"
METADATA_SEPARATOR = ";"

RANDOM_SEED = 42
REMOVE_ZERO_GENES = True
MIN_GENE_VARIANCE = None

print(f"Project root: {PROJECT_ROOT}")

CELLTYPE_COLORS = {
    "neurons": "#D35400",
    "exc_neurons": "#E67E22",
    "inh_neurons": "#F39C12",
    "M/M": "#2E8B57",
    "microglia": "#006D5B",
    "macrophages": "#455A64",
    "hom_mg": "#00A087",
    "inf_mg": "#91D1C2",
    "prol_mg": "#008280",
    "prol_infil_mg": "#4DBBD5",
    "oligodendrocytes": "#5E548E",
    "opc": "#9F86C0",
    "astrocytes": "#BE95C4",
    "endothelial_cells": "#2B4162",
    "epithelial_cells": "#4A6FA5",
    "fibroblasts": "#D6CCC2",
    "b_cells": "#631879",
    "t_cells": "#A20056",
}

CELLTYPE_ORDER = [
    "exc_neurons",
    "inh_neurons",
    "neurons",
    "M/M",
    "hom_mg",
    "inf_mg",
    "prol_mg",
    "prol_infil_mg",
    "microglia",
    "macrophages",
    "oligodendrocytes",
    "opc",
    "astrocytes",
    "endothelial_cells",
    "epithelial_cells",
    "fibroblasts",
    "b_cells",
    "t_cells",
]
DEFAULT_CELLTYPE_COLOR = "#9E9E9E"


def standardize_celltype_label(cell_type: str) -> str:
    key = str(cell_type).strip()
    key = key.replace("-", "_").replace(" ", "_")
    key = "_".join(part for part in key.split("_") if part)
    lower_key = key.lower()
    aliases = {
        "m/m": "M/M",
        "m_m": "M/M",
        "mm": "M/M",
        "b_cell": "b_cells",
        "b_cells": "b_cells",
        "t_cell": "t_cells",
        "t_cells": "t_cells",
        "opc": "opc",
        "opcs": "opc",
        "astrocyte": "astrocytes",
        "astrocytes": "astrocytes",
        "microglia": "microglia",
        "macrophage": "macrophages",
        "macrophages": "macrophages",
        "endothelial_cell": "endothelial_cells",
        "endothelial_cells": "endothelial_cells",
        "epithelial_cell": "epithelial_cells",
        "epithelial_cells": "epithelial_cells",
        "fibroblast": "fibroblasts",
        "fibroblasts": "fibroblasts",
        "neuron": "neurons",
        "neurons": "neurons",
    }
    return aliases.get(lower_key, lower_key)


def order_celltypes(cell_types: Sequence[str]) -> list[str]:
    order_rank = {cell_type: idx for idx, cell_type in enumerate(CELLTYPE_ORDER)}
    return sorted(
        [str(cell_type) for cell_type in cell_types],
        key=lambda cell_type: (
            order_rank.get(standardize_celltype_label(cell_type), len(order_rank)),
            standardize_celltype_label(cell_type),
            str(cell_type),
        ),
    )


def celltype_colors_for(cell_types: Sequence[str]) -> dict[str, str]:
    return {
        str(cell_type): CELLTYPE_COLORS.get(standardize_celltype_label(cell_type), DEFAULT_CELLTYPE_COLOR)
        for cell_type in cell_types
    }


Project root: /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv


## Load counts and metadata


In [37]:
def require_file(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {label}: {path}\n"
            "Check that the data folder is present locally and that the path constants above match your files."
        )


def load_h5ad_data(file_path: str | Path) -> ad.AnnData:
    path = Path(file_path)
    require_file(path, "h5ad counts file")
    if path.suffix.lower() != ".h5ad":
        print(f"Warning: expected a .h5ad file, got {path.suffix}")
    data = ad.read_h5ad(path)
    if not data.obs_names.is_unique:
        raise ValueError("AnnData obs_names are not unique. Cell IDs must be unique before training.")
    if not data.var_names.is_unique:
        print("Gene names are not unique. Making var_names unique before filtering.")
        data.var_names_make_unique()
    print(f"Loaded {path.name}: {data.n_obs:,} cells x {data.n_vars:,} genes")
    return data


def load_cell_metadata(
    file_path: str | Path,
    sep: str = ",",
    cell_id_column: str = CELL_ID_COLUMN,
    celltype_column: str = CELLTYPE_COLUMN,
) -> pd.DataFrame:
    path = Path(file_path)
    require_file(path, "cell metadata file")
    metadata = pd.read_csv(path, sep=sep, index_col=False)
    required = {cell_id_column, celltype_column}
    missing = required.difference(metadata.columns)
    if missing:
        raise ValueError(
            f"Missing metadata columns: {sorted(missing)}. Available columns: {list(metadata.columns)}"
        )
    metadata = metadata.copy()
    metadata[cell_id_column] = metadata[cell_id_column].astype(str)
    metadata[celltype_column] = metadata[celltype_column].astype(str)
    if metadata[cell_id_column].duplicated().any():
        duplicated = metadata.loc[metadata[cell_id_column].duplicated(), cell_id_column].head(5).tolist()
        raise ValueError(f"Metadata contains duplicated cell IDs, for example: {duplicated}")
    print(f"Loaded metadata: {metadata.shape[0]:,} cells x {metadata.shape[1]:,} columns")
    return metadata


In [38]:
adata = load_h5ad_data(COUNTS_PATH)
metadata = load_cell_metadata(METADATA_PATH, sep=METADATA_SEPARATOR)
metadata.head()


Loaded counts.h5ad: 60,482 cells x 15,774 genes
Loaded metadata: 60,482 cells x 10 columns


,cellID,sample,copy_of_sample,main_celltype_luee,sub_celltype_luee,dataset,nCount_RNA,nFeature_RNA,celltype,sub_celltype
0,csf1r_10.Group5.250919_10.Group5.250919_AAACGC...,10.Group5.250919,10.Group5.250919,Microglia,Inf MG,CSF1R,6525.0,2177.0,M/M,inf_mg
1,csf1r_10.Group5.250919_10.Group5.250919_AAACGT...,10.Group5.250919,10.Group5.250919,Microglia,Hom MG,CSF1R,5357.0,2057.0,M/M,hom_mg
2,csf1r_10.Group5.250919_10.Group5.250919_AAACGT...,10.Group5.250919,10.Group5.250919,Microglia,Hom MG,CSF1R,6520.0,2394.0,M/M,hom_mg
3,csf1r_10.Group5.250919_10.Group5.250919_AAACTC...,10.Group5.250919,10.Group5.250919,Microglia,Inf MG,CSF1R,7265.0,2665.0,M/M,inf_mg
4,csf1r_10.Group5.250919_10.Group5.250919_AAAGCA...,10.Group5.250919,10.Group5.250919,Microglia,Inf MG,CSF1R,6075.0,2247.0,M/M,inf_mg


## Align metadata and filter genes

In [39]:
def align_metadata_to_adata(
    adata: ad.AnnData,
    metadata: pd.DataFrame,
    cell_id_column: str = CELL_ID_COLUMN,
    celltype_column: str = CELLTYPE_COLUMN,
) -> ad.AnnData:
    adata = adata.copy()
    adata.obs_names = adata.obs_names.astype(str)
    metadata = metadata.copy()
    metadata[cell_id_column] = metadata[cell_id_column].astype(str)
    metadata[celltype_column] = metadata[celltype_column].astype(str)

    adata_ids = pd.Index(adata.obs_names)
    metadata_ids = pd.Index(metadata[cell_id_column])
    common_ids = adata_ids[adata_ids.isin(metadata_ids)]
    if len(common_ids) == 0:
        raise ValueError("No matching cell IDs were found between AnnData obs_names and metadata cellID.")
    if len(common_ids) != adata.n_obs or len(common_ids) != len(metadata_ids):
        print("Cell ID mismatch detected. Keeping only cells present in both files.")
        print(f"Cells in AnnData: {adata.n_obs:,}")
        print(f"Cells in metadata: {len(metadata_ids):,}")
        print(f"Common cells used: {len(common_ids):,}")

    adata = adata[common_ids, :].copy()
    metadata_indexed = metadata.set_index(cell_id_column).loc[list(adata.obs_names)]
    adata.obs = metadata_indexed.copy()
    adata.obs[celltype_column] = adata.obs[celltype_column].astype(str)
    cell_counts = adata.obs[celltype_column].value_counts().sort_index()
    print(f"Aligned data: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
    display(cell_counts.to_frame("n_cells"))
    return adata


def extract_gene_list_from_json_payload(payload, source_name: str) -> list[str]:
    if isinstance(payload, list):
        raw_genes = payload
    elif isinstance(payload, dict):
        list_values = [value for value in payload.values() if isinstance(value, list)]
        if not list_values:
            raise ValueError(f"{source_name} must contain a list of genes or a dictionary with a list value.")
        raw_genes = max(list_values, key=len)
    else:
        raise ValueError(f"{source_name} must contain a list of genes or a dictionary with a list value.")

    genes = pd.Series(raw_genes, dtype="object").dropna().astype(str).str.strip()
    genes = genes[genes.ne("")]
    return genes.drop_duplicates().tolist()


def load_json_gene_list(file_path: str | Path, label: str = "gene JSON") -> list[str]:
    path = Path(file_path)
    require_file(path, label)
    with path.open(encoding="utf-8") as handle:
        payload = json.load(handle)
    genes = extract_gene_list_from_json_payload(payload, path.name)
    print(f"Loaded {len(genes):,} genes from {path.name}")
    return genes


def load_excluded_de_genes(
    sc_de_path: str | Path = SC_DE_GENE_PATH,
    sn_de_path: str | Path = SN_DE_GENE_PATH,
) -> list[str]:
    sc_de_genes = load_json_gene_list(sc_de_path, label="single-cell DE gene JSON")
    sn_de_genes = load_json_gene_list(sn_de_path, label="single-nucleus DE gene JSON")
    excluded = sorted(set(sc_de_genes).union(sn_de_genes))
    print(f"Using {len(excluded):,} total DE genes to remove from reference and bulk.")
    return excluded


def load_common_bulk_genes(file_path: str | Path) -> list[str]:
    path = Path(file_path)
    require_file(path, "common bulk gene JSON")
    with path.open(encoding="utf-8") as handle:
        payload = json.load(handle)

    if isinstance(payload, list):
        raw_genes = payload
    elif isinstance(payload, dict):
        list_values = [value for value in payload.values() if isinstance(value, list)]
        if not list_values:
            raise ValueError(f"{path.name} must contain a list of genes or a dictionary with a list value.")
        raw_genes = max(list_values, key=len)
    else:
        raise ValueError(f"{path.name} must contain a list of genes or a dictionary with a list value.")

    genes = pd.Series(raw_genes, dtype="object").dropna().astype(str).str.strip()
    genes = genes[genes.ne("")]
    unique_genes = genes.drop_duplicates().tolist()
    print(f"Loaded {len(unique_genes):,} common bulk genes from {path.name}")
    return unique_genes


def matrix_mean_and_variance(X) -> tuple[np.ndarray, np.ndarray]:
    means = np.asarray(X.mean(axis=0)).ravel()
    if hasattr(X, "multiply"):
        second_moments = np.asarray(X.multiply(X).mean(axis=0)).ravel()
    else:
        second_moments = np.asarray(np.square(X).mean(axis=0)).ravel()
    variances = np.maximum(second_moments - np.square(means), 0.0)
    return means, variances


def filter_feature_genes(
    adata: ad.AnnData,
    genes_to_remove: Iterable[str],
    remove_zero_genes: bool = True,
    min_gene_variance: float | None = None,
    case_sensitive: bool = True,
) -> ad.AnnData:
    adata = adata.copy()
    gene_names = pd.Index(adata.var_names.astype(str))
    if case_sensitive:
        removal_set = {str(g).strip() for g in genes_to_remove if str(g).strip()}
        removal_mask = gene_names.isin(removal_set)
    else:
        removal_set = {str(g).strip().upper() for g in genes_to_remove if str(g).strip()}
        removal_mask = gene_names.str.upper().isin(removal_set)

    keep_mask = ~np.asarray(removal_mask)
    report = {"genes_start": adata.n_vars, "technology_driven_DE_genes_removed": int(removal_mask.sum())}
    if remove_zero_genes or min_gene_variance is not None:
        means, variances = matrix_mean_and_variance(adata.X)
        if remove_zero_genes:
            zero_mask = means <= 0
            keep_mask &= ~zero_mask
            report["genes_removed_zero_expression"] = int(zero_mask.sum())
        if min_gene_variance is not None:
            low_variance_mask = variances < min_gene_variance
            keep_mask &= ~low_variance_mask
            report["genes_removed_low_variance"] = int(low_variance_mask.sum())

    filtered = adata[:, keep_mask].copy()
    report["genes_kept"] = filtered.n_vars
    display(pd.Series(report, name="gene_filtering"))
    return filtered


def filter_to_common_bulk_genes(
    adata: ad.AnnData,
    common_bulk_genes: Iterable[str],
    case_sensitive: bool = True,
) -> ad.AnnData:
    adata = adata.copy()
    reference_genes = pd.Index(adata.var_names.astype(str))
    if case_sensitive:
        common_set = {str(g).strip() for g in common_bulk_genes if str(g).strip()}
        keep_mask = reference_genes.isin(common_set)
    else:
        common_set = {str(g).strip().upper() for g in common_bulk_genes if str(g).strip()}
        keep_mask = reference_genes.str.upper().isin(common_set)

    n_kept = int(np.asarray(keep_mask).sum())
    if n_kept == 0:
        raise ValueError("No genes overlap between the filtered reference and common_bulk_genes.json.")

    filtered = adata[:, np.asarray(keep_mask)].copy()
    report = pd.Series(
        {
            "reference_genes_before_common_bulk_filter": adata.n_vars,
            "common_bulk_genes_loaded": len(common_set),
            "genes_kept_in_reference_bulk_intersection": filtered.n_vars,
            "reference_genes_removed_not_in_common_bulk": adata.n_vars - filtered.n_vars,
        },
        name="common_bulk_gene_filtering",
    )
    display(report)
    return filtered


In [40]:
adata = align_metadata_to_adata(adata, metadata)
excluded_de_genes = load_excluded_de_genes(SC_DE_GENE_PATH, SN_DE_GENE_PATH)
adata = filter_feature_genes(
    adata,
    excluded_de_genes,
    remove_zero_genes=REMOVE_ZERO_GENES,
    min_gene_variance=MIN_GENE_VARIANCE,
    case_sensitive=True,
)
common_bulk_genes = load_common_bulk_genes(COMMON_BULK_GENES_PATH)
adata = filter_to_common_bulk_genes(
    adata,
    common_bulk_genes,
    case_sensitive=True,
)
adata


Aligned data: 60,482 cells x 15,774 genes


,n_cells
sub_celltype,
astrocytes,4935
b_cells,309
endothelial_cells,4607
epithelial_cells,42
exc_neurons,10011
fibroblasts,89
hom_mg,6741
inf_mg,3959
inh_neurons,10839


Loaded 0 genes from empty.json
Loaded 0 genes from empty.json
Using 0 total DE genes to remove from reference and bulk.


genes_start                           15774
technology_driven_DE_genes_removed        0
genes_removed_zero_expression             6
genes_kept                            15768
Name: gene_filtering, dtype: int64

Loaded 14,926 common bulk genes from common_bulk_genes.json


reference_genes_before_common_bulk_filter     15768
common_bulk_genes_loaded                      14926
genes_kept_in_reference_bulk_intersection     13685
reference_genes_removed_not_in_common_bulk     2083
Name: common_bulk_gene_filtering, dtype: int64

AnnData object with n_obs × n_vars = 60482 × 13685
    obs: 'sample', 'copy_of_sample', 'main_celltype_luee', 'sub_celltype_luee', 'dataset', 'nCount_RNA', 'nFeature_RNA', 'celltype', 'sub_celltype'
    uns: 'X_name'

## Split cells by cell type


In [41]:
def split_cells_by_type(
    adata: ad.AnnData,
    train_fraction: float = 0.8,
    celltype_column: str = CELLTYPE_COLUMN,
    seed: int = RANDOM_SEED,
) -> tuple[ad.AnnData, ad.AnnData, pd.DataFrame]:
    if not 0 < train_fraction < 1:
        raise ValueError("train_fraction must be between 0 and 1.")
    rng = np.random.default_rng(seed)
    obs = adata.obs.reset_index(drop=True)
    train_positions = []
    test_positions = []
    rows = []
    for cell_type, group in obs.groupby(celltype_column, observed=True, sort=True):
        positions = group.index.to_numpy(dtype=int)
        rng.shuffle(positions)
        n_cells = len(positions)
        if n_cells == 1:
            n_train = 1
        else:
            n_train = int(round(n_cells * train_fraction))
            n_train = max(1, min(n_cells - 1, n_train))
        train_positions.append(positions[:n_train])
        test_positions.append(positions[n_train:])
        rows.append(
            {
                "celltype": cell_type,
                "total_cells": n_cells,
                "train_cells": n_train,
                "test_cells": n_cells - n_train,
            }
        )

    train_idx = np.concatenate(train_positions)
    test_idx = np.concatenate([idx for idx in test_positions if len(idx) > 0])
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    train_adata = adata[train_idx, :].copy()
    test_adata = adata[test_idx, :].copy()
    split_report = pd.DataFrame(rows).sort_values("celltype").reset_index(drop=True)
    if (split_report["test_cells"] == 0).any():
        missing = split_report.loc[split_report["test_cells"] == 0, "celltype"].tolist()
        print(f"Warning: these cell types have no held-out test cells: {missing}")
    print(f"Train cells: {train_adata.n_obs:,}")
    print(f"Test cells: {test_adata.n_obs:,}")
    display(split_report)
    return train_adata, test_adata, split_report


In [42]:
available_cell_types = adata.obs[CELLTYPE_COLUMN].astype(str).unique().tolist()
CELL_TYPES = order_celltypes(available_cell_types)
CELLTYPE_PLOT_COLORS = celltype_colors_for(CELL_TYPES)

display(pd.DataFrame({"celltype": CELL_TYPES, "style_key": [standardize_celltype_label(ct) for ct in CELL_TYPES], "color": [CELLTYPE_PLOT_COLORS[ct] for ct in CELL_TYPES]}))
CELL_TYPES

train_adata, test_adata, split_report = split_cells_by_type(adata)


,celltype,style_key,color
0,exc_neurons,exc_neurons,#E67E22
1,inh_neurons,inh_neurons,#F39C12
2,hom_mg,hom_mg,#00A087
3,inf_mg,inf_mg,#91D1C2
4,prol_mg,prol_mg,#008280
5,macrophages,macrophages,#455A64
6,oligodendrocytes,oligodendrocytes,#5E548E
7,opc,opc,#9F86C0
8,astrocytes,astrocytes,#BE95C4
9,endothelial_cells,endothelial_cells,#2B4162


Train cells: 48,385
Test cells: 12,097


,celltype,total_cells,train_cells,test_cells
0,astrocytes,4935,3948,987
1,b_cells,309,247,62
2,endothelial_cells,4607,3686,921
3,epithelial_cells,42,34,8
4,exc_neurons,10011,8009,2002
5,fibroblasts,89,71,18
6,hom_mg,6741,5393,1348
7,inf_mg,3959,3167,792
8,inh_neurons,10839,8671,2168
9,macrophages,5909,4727,1182


## Generate and save pseudobulks


In [43]:
def sum_expression_rows(X, row_indices: np.ndarray) -> np.ndarray:
    subset = X[row_indices]
    summed = subset.sum(axis=0)
    return np.asarray(summed).ravel().astype(np.float32)


def build_dirichlet_alpha_schedule(
    n_samples: int,
    dirichlet_alpha_values: float | Sequence[float],
    seed: int = RANDOM_SEED,
) -> tuple[np.ndarray, pd.DataFrame]:
    alpha_values = np.atleast_1d(np.asarray(dirichlet_alpha_values, dtype=np.float64))
    if alpha_values.ndim != 1 or len(alpha_values) == 0:
        raise ValueError("dirichlet_alpha_values must be one value or a one-dimensional list of values.")
    if np.any(alpha_values <= 0):
        raise ValueError("All Dirichlet alpha values must be positive.")

    n_groups = len(alpha_values)
    group_sizes = np.full(n_groups, n_samples // n_groups, dtype=int)
    group_sizes[: n_samples % n_groups] += 1
    schedule = np.concatenate([np.repeat(alpha, size) for alpha, size in zip(alpha_values, group_sizes)])
    rng = np.random.default_rng(seed)
    rng.shuffle(schedule)
    summary = pd.DataFrame({"dirichlet_alpha": alpha_values, "n_samples": group_sizes})
    return schedule.astype(np.float64), summary


def make_pseudobulks(
    adata: ad.AnnData,
    n_samples: int,
    cells_per_sample: int,
    cell_types: Sequence[str],
    celltype_column: str = CELLTYPE_COLUMN,
    dirichlet_alpha_values: float | Sequence[float] = 1.0,
    seed: int = RANDOM_SEED,
    verbose: bool = True,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    if n_samples <= 0:
        raise ValueError("n_samples must be positive.")
    if cells_per_sample <= 0:
        raise ValueError("cells_per_sample must be positive.")

    rng = np.random.default_rng(seed)
    alpha_schedule, alpha_summary = build_dirichlet_alpha_schedule(n_samples, dirichlet_alpha_values, seed=seed)
    display(alpha_summary)

    obs_celltypes = adata.obs[celltype_column].astype(str).to_numpy()
    cell_types = [str(ct) for ct in cell_types]
    celltype_to_column = {ct: idx for idx, ct in enumerate(cell_types)}
    indices_by_type = {ct: np.flatnonzero(obs_celltypes == ct) for ct in cell_types}
    available_types = [ct for ct in cell_types if len(indices_by_type[ct]) > 0]
    if not available_types:
        raise ValueError("No requested cell types are available in this AnnData object.")

    bulk_matrix = np.zeros((n_samples, adata.n_vars), dtype=np.float32)
    proportions = np.zeros((n_samples, len(cell_types)), dtype=np.float32)
    sample_ids = [f"pb_{i:05d}" for i in range(n_samples)]
    progress_every = max(1, n_samples // 5)

    for sample_idx, alpha_value in enumerate(alpha_schedule):
        alpha = np.full(len(available_types), float(alpha_value), dtype=np.float64)
        target_fractions = rng.dirichlet(alpha)
        sampled_counts = rng.multinomial(cells_per_sample, target_fractions)
        chosen_chunks = []

        for cell_type, n_cells in zip(available_types, sampled_counts):
            if n_cells == 0:
                continue
            selected = rng.choice(indices_by_type[cell_type], size=int(n_cells), replace=True)
            chosen_chunks.append(selected)
            proportions[sample_idx, celltype_to_column[cell_type]] = n_cells / cells_per_sample

        chosen_indices = np.concatenate(chosen_chunks)
        bulk_matrix[sample_idx, :] = sum_expression_rows(adata.X, chosen_indices)

        if verbose and (sample_idx + 1) % progress_every == 0:
            print(f"Generated {sample_idx + 1:,}/{n_samples:,} pseudobulks")

    sample_metadata = pd.DataFrame({"sample_id": sample_ids, "dirichlet_alpha": alpha_schedule})
    return bulk_matrix, proportions, sample_metadata


def save_pseudobulk_dataset(
    output_dir: str | Path,
    prefix: str,
    X: np.ndarray,
    y: np.ndarray,
    genes: Sequence[str],
    cell_types: Sequence[str],
    sample_metadata: pd.DataFrame,
) -> dict[str, Path]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    npz_path = output_dir / f"{prefix}_pseudobulk.npz"
    proportions_path = output_dir / f"{prefix}_proportions.csv"
    summary_path = output_dir / f"{prefix}_summary.csv"
    metadata_path = output_dir / f"{prefix}_metadata.csv"

    sample_metadata = sample_metadata.copy()
    sample_ids = sample_metadata["sample_id"].astype(str).to_numpy()
    np.savez_compressed(
        npz_path,
        X=X.astype(np.float32, copy=False),
        y=y.astype(np.float32, copy=False),
        genes=np.asarray(list(genes), dtype=str),
        cell_types=np.asarray(list(cell_types), dtype=str),
        sample_ids=sample_ids,
        dirichlet_alpha=sample_metadata["dirichlet_alpha"].to_numpy(dtype=np.float32),
    )
    pd.DataFrame(y, index=sample_ids, columns=cell_types).to_csv(proportions_path, index_label="sample_id")
    sample_metadata.to_csv(metadata_path, index=False)
    summary = sample_metadata.copy()
    summary["total_counts"] = X.sum(axis=1)
    summary["detected_genes"] = (X > 0).sum(axis=1)
    summary.to_csv(summary_path, index=False)
    print(f"Saved {prefix} pseudobulks to {npz_path}")
    print(f"Saved {prefix} proportions to {proportions_path}")
    return {"npz": npz_path, "proportions": proportions_path, "summary": summary_path, "metadata": metadata_path}


def safe_filename(text: str) -> str:
    safe = "".join(ch if ch.isalnum() else "_" for ch in str(text)).strip("_")
    return safe or "celltype"


def plot_pseudobulk_proportion_distributions(
    y: np.ndarray,
    cell_types: Sequence[str],
    sample_metadata: pd.DataFrame,
    output_dir: str | Path,
    prefix: str,
    bins: int = 30,
    density: bool = True,
    plot_kind: str = "histogram",  # "histogram" or "density"
) -> list[Path]:
    """
    Plot pseudobulk simulated proportions per cell type.

    Parameters
    ----------
    plot_kind:
        "histogram" plots histograms.
        "density" plots kernel density curves.
    """

    if plot_kind not in {"histogram", "density"}:
        raise ValueError("plot_kind must be either 'histogram' or 'density'")

    output_dir = Path(output_dir) / prefix
    output_dir.mkdir(parents=True, exist_ok=True)

    proportion_df = pd.DataFrame(y, columns=cell_types)
    proportion_df["dirichlet_alpha"] = sample_metadata["dirichlet_alpha"].to_numpy()

    alpha_values = sorted(proportion_df["dirichlet_alpha"].unique())
    saved_paths = []

    for cell_type in cell_types:
        fig, ax = plt.subplots(figsize=(6, 4))

        for alpha_value in alpha_values:
            values = proportion_df.loc[
                proportion_df["dirichlet_alpha"] == alpha_value,
                cell_type,
            ].dropna()

            if plot_kind == "histogram":
                ax.hist(
                    values,
                    bins=bins,
                    density=density,
                    alpha=0.45,
                    label=f"alpha={alpha_value:g}",
                )

            elif plot_kind == "density":
                values.plot.kde(
                    ax=ax,
                    label=f"alpha={alpha_value:g}",
                )

        ax.set_title(f"{prefix} simulated proportion: {cell_type}")
        ax.set_xlabel("Simulated proportion")

        if plot_kind == "histogram":
            ax.set_ylabel("Density" if density else "Count")
        else:
            ax.set_ylabel("Density")

        ax.set_xlim(0, 1)

        if len(alpha_values) > 1:
            ax.legend(frameon=False)

        fig.tight_layout()

        output_path = output_dir / (
            f"{safe_filename(cell_type)}_proportion_{plot_kind}.png"
        )

        fig.savefig(output_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        saved_paths.append(output_path)

    print(
        f"Saved {len(saved_paths)} {prefix} proportion {plot_kind} plots to {output_dir}"
    )

    return saved_paths


In [44]:
N_TRAIN_PSEUDOBULKS = 10000
N_TEST_PSEUDOBULKS = 2500
CELLS_PER_PSEUDOBULK = 500
DIRICHLET_ALPHA_VALUES = [0.1, 0.3, 1.0, 5.0]

train_pb_X, train_pb_y, train_sample_metadata = make_pseudobulks(
    train_adata,
    n_samples=N_TRAIN_PSEUDOBULKS,
    cells_per_sample=CELLS_PER_PSEUDOBULK,
    cell_types=CELL_TYPES,
    dirichlet_alpha_values=DIRICHLET_ALPHA_VALUES,
    seed=RANDOM_SEED,
)
test_pb_X, test_pb_y, test_sample_metadata = make_pseudobulks(
    test_adata,
    n_samples=N_TEST_PSEUDOBULKS,
    cells_per_sample=CELLS_PER_PSEUDOBULK,
    cell_types=CELL_TYPES,
    dirichlet_alpha_values=DIRICHLET_ALPHA_VALUES,
    seed=RANDOM_SEED + 1,
)

train_paths = save_pseudobulk_dataset(
    PSEUDOBULK_DIR, "train", train_pb_X, train_pb_y, adata.var_names.astype(str).tolist(), CELL_TYPES, train_sample_metadata
)
test_paths = save_pseudobulk_dataset(
    PSEUDOBULK_DIR, "test", test_pb_X, test_pb_y, adata.var_names.astype(str).tolist(), CELL_TYPES, test_sample_metadata
)

# Histograms
train_plot_paths = plot_pseudobulk_proportion_distributions(train_pb_y, CELL_TYPES, train_sample_metadata, PLOTS_DIR, "train", plot_kind= "histogram")
test_plot_paths = plot_pseudobulk_proportion_distributions(test_pb_y, CELL_TYPES, test_sample_metadata, PLOTS_DIR, "test", plot_kind= "histogram")

# Densitograms
train_plot_paths = plot_pseudobulk_proportion_distributions(train_pb_y, CELL_TYPES, train_sample_metadata, PLOTS_DIR, "train", plot_kind= "density")
test_plot_paths = plot_pseudobulk_proportion_distributions(test_pb_y, CELL_TYPES, test_sample_metadata, PLOTS_DIR, "test", plot_kind= "density")

train_sample_ids = train_sample_metadata["sample_id"].tolist()
test_sample_ids = test_sample_metadata["sample_id"].tolist()
print(f"Train pseudobulk matrix: {train_pb_X.shape}")
print(f"Test pseudobulk matrix: {test_pb_X.shape}")
pd.DataFrame(train_pb_y[:5], index=train_sample_ids[:5], columns=CELL_TYPES)


,dirichlet_alpha,n_samples
0,0.1,2500
1,0.3,2500
2,1.0,2500
3,5.0,2500


Generated 2,000/10,000 pseudobulks
Generated 4,000/10,000 pseudobulks
Generated 6,000/10,000 pseudobulks
Generated 8,000/10,000 pseudobulks
Generated 10,000/10,000 pseudobulks


,dirichlet_alpha,n_samples
0,0.1,625
1,0.3,625
2,1.0,625
3,5.0,625


Generated 500/2,500 pseudobulks
Generated 1,000/2,500 pseudobulks
Generated 1,500/2,500 pseudobulks
Generated 2,000/2,500 pseudobulks
Generated 2,500/2,500 pseudobulks
Saved train pseudobulks to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/rat_sub_std/pseudobulk/train_pseudobulk.npz
Saved train proportions to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/rat_sub_std/pseudobulk/train_proportions.csv
Saved test pseudobulks to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/rat_sub_std/pseudobulk/test_pseudobulk.npz
Saved test proportions to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/rat_sub_std/pseudobulk/test_proportions.csv
Saved 14 train proportion histogram plots to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/rat_sub_st

,exc_neurons,inh_neurons,hom_mg,inf_mg,prol_mg,macrophages,oligodendrocytes,opc,astrocytes,endothelial_cells,epithelial_cells,fibroblasts,b_cells,t_cells
pb_00000,0.076,0.066,0.060,0.052,0.066,0.064,0.094,0.072,0.092,0.034,0.118,0.042,0.090,0.074
pb_00001,0.066,0.036,0.060,0.046,0.112,0.124,0.080,0.086,0.046,0.054,0.036,0.062,0.088,0.104
pb_00002,0.708,0.048,0.000,0.200,0.000,0.000,0.004,0.026,0.012,0.000,0.000,0.000,0.002,0.000
pb_00003,0.076,0.068,0.036,0.048,0.064,0.116,0.060,0.038,0.082,0.106,0.154,0.046,0.034,0.072
pb_00004,0.014,0.000,0.472,0.000,0.038,0.000,0.004,0.004,0.004,0.002,0.336,0.026,0.016,0.084


In [45]:
split_report.to_csv(PSEUDOBULK_DIR / "cell_split_report.csv", index=False)
pd.DataFrame(
    {
        "celltype": CELL_TYPES,
        "style_key": [standardize_celltype_label(ct) for ct in CELL_TYPES],
        "color": [CELLTYPE_PLOT_COLORS[ct] for ct in CELL_TYPES],
    }
).to_csv(PSEUDOBULK_DIR / "celltype_order_and_colors.csv", index=False)
print(f"Saved pseudobulk manifest files to {PSEUDOBULK_DIR}")


Saved pseudobulk manifest files to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/rat_sub_std/pseudobulk
